####Imports & Intallations

In [ ]:
!pip install -q kagglehub pandas numpy matplotlib seaborn plotly
!pip install -q scikit-learn umap-learn hdbscan
!pip install -q sentence-transformers transformers torch
!pip install -q bertopic wordcloud nltk spacy
!pip install -q streamlit plotly-express deep-translator
!pip install -q langdetect textblob

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import pickle
from pathlib import Path

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

##Section 2: Directory Structure Setup

In [ ]:
main_directory = '/content/drive/MyDrive/Barnabus Project/'
directories = [
    f'{main_directory}data/raw',
    f'{main_directory}data/processed',
    f'{main_directory}data/embeddings',
    f'{main_directory}outputs/figures',
    f'{main_directory}outputs/recommendations',
    f'{main_directory}models/cache'
]

In [ ]:
for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {directory}")

##Section 3: Load Dataset

In [46]:
import kagglehub

path = kagglehub.dataset_download("thedevastator/german-2021-patient-reviews-and-ratings-of-docto")
print(f"Dataset downloaded to: {path}")

import glob
csv_files = glob.glob(f"{path}/**/*.csv", recursive=True)
for f in csv_files:
    print(f"  - {f}")

df_raw = pd.read_csv(csv_files[0])

df_raw.to_csv(f'{main_directory}data/raw/original_data.csv', index=False)

Using Colab cache for faster access to the 'german-2021-patient-reviews-and-ratings-of-docto' dataset.
Dataset downloaded to: /kaggle/input/german-2021-patient-reviews-and-ratings-of-docto
  - /kaggle/input/german-2021-patient-reviews-and-ratings-of-docto/2021_german_doctor_reviews.csv


##Section 4: Initial Data Exploration

In [47]:
print("DATASET INFO:")
print(f"Total rows: {len(df_raw):,}")
print(f"Total columns: {len(df_raw.columns)}")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("COLUMN NAMES:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:2d}. {col}")

print("FIRST 3 ROWS:")
print(df_raw.head(3))

print("DATA TYPES:")
print(df_raw.dtypes)

print("MISSING VALUES:")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))

print("NUMERIC COLUMNS SUMMARY:")
print(df_raw.describe())

DATASET INFO:
Total rows: 439,280
Total columns: 3
Memory usage: 191.81 MB
COLUMN NAMES:
 1. index
 2. rating
 3. comment
FIRST 3 ROWS:
   index  rating                                            comment
0      0     2.0  Ich bin franzose und bin seit ein paar Wochen ...
1      1     6.0  Dieser Arzt ist das unmöglichste was mir in me...
2      2     1.0  Hatte akute Beschwerden am Rücken. Herr Magura...
DATA TYPES:
index        int64
rating     float64
comment     object
dtype: object
MISSING VALUES:
        Missing Count  Percentage
rating           9544        2.17
NUMERIC COLUMNS SUMMARY:
               index         rating
count  439280.000000  429736.000000
mean   219639.500000       1.560684
std    126809.357461       1.344886
min         0.000000       1.000000
25%    109819.750000       1.000000
50%    219639.500000       1.000000
75%    329459.250000       1.000000
max    439279.000000       6.000000


In [48]:
text_columns = []
for col in df_raw.columns:
    if df_raw[col].dtype == 'object':
        avg_length = df_raw[col].dropna().astype(str).str.len().mean()
        if avg_length > 20:
            text_columns.append(col)
            print(f"Text column detected: '{col}'")
            print(f"   Average length: {avg_length:.0f} characters")
            print(f"   Sample: {df_raw[col].dropna().iloc[0][:100]}...")

Text column detected: 'comment'
   Average length: 366 characters
   Sample: Ich bin franzose und bin seit ein paar Wochen in muenchen. Ich hatte Zahn Schmerzen und mein Kollegu...


In [49]:
rating_columns = []
for col in df_raw.columns:
    if 'rating' in col.lower() or 'score' in col.lower() or 'stars' in col.lower():
        rating_columns.append(col)
        print(f"Rating column detected: '{col}'")
        print(df_raw[col].value_counts().sort_index())

Rating column detected: 'rating'
rating
1.0    349813
2.0     22151
3.0      8660
4.0     12269
5.0     19547
6.0     17296
Name: count, dtype: int64


In [52]:
df_raw.dropna(inplace=True)
df_raw.reset_index(drop=True, inplace=True)
df_raw.head()

,index,rating,comment
0,0,2.0,Ich bin franzose und bin seit ein paar Wochen ...
1,1,6.0,Dieser Arzt ist das unmöglichste was mir in me...
2,2,1.0,Hatte akute Beschwerden am Rücken. Herr Magura...
3,3,1.0,Nachdem ich in der Klinik nur ungenaue Angaben...
4,4,1.0,"Frau Dr. Vetter kenne ich seit vielen Jahren, ..."


In [53]:
len(df_raw)

429736

##Section 5: Identify Key Columns

In [54]:
column_mapping = {
    'text_columns': text_columns,
    'rating_columns': rating_columns,
}

with open(f'{main_directory}data/processed/column_mapping.json', 'w') as f:
    json.dump(column_mapping, f, indent=2)

##Section 6: Visualization

In [55]:
# 1. Missing data heatmap
if missing.sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(df_raw.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Missing Data Heatmap', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{main_directory}outputs/figures/01_missing_data_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()

In [56]:
# 2. Rating distribution (if available)
if rating_columns:
    fig, axes = plt.subplots(1, len(rating_columns), figsize=(6*len(rating_columns), 5))
    if len(rating_columns) == 1:
        axes = [axes]

    for idx, col in enumerate(rating_columns):
        df_raw[col].value_counts().sort_index().plot(kind='bar', ax=axes[idx])
        axes[idx].set_title(f'Distribution: {col}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Rating')
        axes[idx].set_ylabel('Count')
        axes[idx].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{main_directory}outputs/figures/02_rating_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()

In [57]:
# 3. Text length distribution
if text_columns:
    fig, axes = plt.subplots(1, len(text_columns), figsize=(8*len(text_columns), 5))
    if len(text_columns) == 1:
        axes = [axes]

    for idx, col in enumerate(text_columns):
        text_lengths = df_raw[col].dropna().astype(str).str.len()
        axes[idx].hist(text_lengths, bins=50, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Text Length Distribution: {col}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Number of Characters')
        axes[idx].set_ylabel('Frequency')
        axes[idx].axvline(text_lengths.mean(), color='red', linestyle='--',
                         label=f'Mean: {text_lengths.mean():.0f}')
        axes[idx].axvline(text_lengths.median(), color='green', linestyle='--',
                         label=f'Median: {text_lengths.median():.0f}')
        axes[idx].legend()
        axes[idx].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{main_directory}outputs/figures/03_text_length_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()

##Section 7: Data Quality Report

In [58]:
import os, json, numpy as np
from pathlib import Path

# If you didn't define this earlier:
# missing = df_raw.isna().sum()

# Build the report (same as before)
quality_report = {
    'total_rows': int(len(df_raw)),
    'total_columns': int(len(df_raw.columns)),
    'missing_values_total': int(missing.sum()),
    'missing_percentage': f"{(missing.sum() / (len(df_raw) * len(df_raw.columns)) * 100):.2f}%",
    'duplicate_rows': int(df_raw.duplicated().sum()),
    'text_columns': list(text_columns),
    'rating_columns': list(rating_columns),
    'memory_mb': f"{df_raw.memory_usage(deep=True).sum() / 1024**2:.2f}"
}

# Nice printout
for key, value in quality_report.items():
    print(f"  {key}: {value}")

# Ensure output directory exists
out_dir = Path(f"{main_directory}outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# Helper to make JSON-safe (handles any leftover NumPy scalars)
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

# Save quality report
with open(out_dir / "data_quality_report.json", "w") as f:
    json.dump(make_json_safe(quality_report), f, indent=2)

print("PHASE 1 COMPLETE: DATA LOADED & EXPLORED")
print("\nNext steps:")
print("1. Review the visualizations in outputs/figures/")
print("2. Proceed to 02_preprocessing_translation.ipynb")
print("3. We will sample 6000 rows with random_state=42")
print("4. Then translate to 4 languages (25% each)")


  total_rows: 429736
  total_columns: 3
  missing_values_total: 9544
  missing_percentage: 0.74%
  duplicate_rows: 0
  text_columns: ['comment']
  rating_columns: ['rating']
  memory_mb: 332.47
PHASE 1 COMPLETE: DATA LOADED & EXPLORED

Next steps:
1. Review the visualizations in outputs/figures/
2. Proceed to 02_preprocessing_translation.ipynb
3. We will sample 6000 rows with random_state=42
4. Then translate to 4 languages (25% each)
